# Catcher: telling humans from bots by *how* they solve a CAPTCHA

Half B of the project. `breaker.ipynb` trains a CNN that reads CAPTCHAs. This notebook looks at the
**behaviour** of whoever solved one (timing, typing rhythm, mouse movement) and decides: human or bot?

The experiment has three rounds, and each one is a row in the final table:

1. **Round 1** - the Catcher learns to spot a *naive* bot (answers instantly, no real mouse).
2. **Round 2** - the Breaker adds an *evasion layer* (fake human timing and a curved mouse move). Does the Catcher still work?
3. **Round 3** - the Catcher fights back: better features (how the cursor moves, not just how long it takes) and retraining.

> **Data warning.** With `USE_SYNTHETIC = True` the notebook runs on **fake** sessions made by
> `simulate_sessions.py`. That only proves the code works. Every number for the report must come from
> real human sessions (`data/human_sessions.csv`) and the Breaker's real logs (`../half_a_breaker/logs/bot_sessions.csv`),
> with `USE_SYNTHETIC = False`.

## Step 1: imports and settings

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from features import load_sessions, BASIC_FEATURES, ADVANCED_FEATURES
import simulate_sessions

USE_SYNTHETIC = True     # True = FAKE test data.  Set to False for the real experiment.
N_REPEATS = 20           # we average over this many different random train/test splits
TEST_FRACTION = 0.3      # share of people / bots held out for testing

HUMAN_CSV = "data/human_sessions.csv"                    # recorded by the capture website
BOT_CSV = "../half_a_breaker/logs/bot_sessions.csv"      # written by the Breaker (shared log format)

# Features to leave out of the "better features" set. See the last cell: if the Breaker logs its mouse at a
# different rhythm than browsers do, ["sample_dt_std_ms"] stops the Catcher from winning on that alone.
EXCLUDE = []
ADVANCED = [f for f in ADVANCED_FEATURES if f not in EXCLUDE]

os.makedirs("results", exist_ok=True)

## Step 2: load the sessions

Both CSV files use the shared log format: one row per event (mouse move, click, key down/up, submit), one session = all rows
with the same `session_id`. `load_sessions` turns each session into a row of numbers with the **same code** for humans and
bots, so a feature always means the same thing.

In [ ]:
if USE_SYNTHETIC:
    simulate_sessions.generate("data/synthetic")          # cheap and repeatable; only touches data/synthetic/
    files = ["data/synthetic/human_sessions.csv", "data/synthetic/bot_sessions.csv"]
    print("USING FAKE DATA - the numbers below are NOT results\n")
else:
    files = [HUMAN_CSV, BOT_CSV]

df = load_sessions(files)
if df.empty:
    raise ValueError(f"No sessions found in {files}. Collect human sessions with the capture website "
                     "and get the Breaker's bot_sessions.csv, or set USE_SYNTHETIC = True to test.")

# humans without real mouse movement (phones, broken logs) would look like bots: drop them
no_mouse = (df.kind == "human") & (df.num_mouse_samples < 3)
if no_mouse.any():
    print(f"Dropped {no_mouse.sum()} human sessions with (almost) no mouse movement")
df = df[~no_mouse].reset_index(drop=True)

print(df.kind.value_counts().to_string())
print("human participants:", df[df.kind == "human"].participant.nunique())

### Checks
We need all three kinds, and enough of each for the numbers to mean something.

In [ ]:
RECOMMENDED = {"human": 100, "naive": 50, "evasive": 50}
for kind, want in RECOMMENDED.items():
    have = int((df.kind == kind).sum())
    if have == 0:
        raise ValueError(f"No '{kind}' sessions found in {files}")
    if have < want:
        print(f"WARNING: only {have} '{kind}' sessions (aim for {want}+). Results will be shaky.")

## Step 3: look at the data

Median of a few features per kind. If a feature has a very different median for humans and bots, it can help the model.

In [ ]:
show = ["total_solve_time_ms", "time_to_first_keystroke_ms", "keystroke_interval_cv", "key_dwell_mean_ms",
        "num_backspaces", "num_mouse_samples", "mouse_path_length_px", "sample_dt_std_ms", "pause_count",
        "straightness", "acceleration_std"]
df.groupby("kind")[show].median().round(2).T

## Step 4: train / test split (by person)

A person's style is personal: if the same person is in both train and test, the model can learn "this is person A's style" instead of
"this is a human", and accuracy looks better than it is. So **whole people** go to the test side.
Bots are split at random (a bot has no personality). The person is read from the human `session_id`, which looks like `h_<initials>_<id>`.

In [ ]:
def split(df, seed, test_fraction=TEST_FRACTION):
    rng = np.random.default_rng(seed)
    test_idx = []

    humans = df[df.kind == "human"]
    people = humans.participant.unique()
    if len(people) >= 3:                                   # split by person
        n_test = max(1, round(len(people) * test_fraction))
        test_people = rng.choice(people, n_test, replace=False)
        test_idx += list(humans.index[humans.participant.isin(test_people)])
    else:                                                  # too few people: fall back to random
        if seed == 0:
            print("WARNING: fewer than 3 participants, splitting humans at random (optimistic results)")
        test_idx += list(rng.choice(humans.index, max(1, int(len(humans) * test_fraction)), replace=False))

    for kind in ["naive", "evasive"]:
        idx = df.index[df.kind == kind]
        test_idx += list(rng.choice(idx, max(1, int(len(idx) * test_fraction)), replace=False))

    return df.drop(test_idx), df.loc[test_idx]

train, test = split(df, seed=0)
print("train:", train.kind.value_counts().to_dict())
print("test: ", test.kind.value_counts().to_dict())

## Step 5: models and one experiment run

`run_once` trains on the chosen kinds with the chosen features and reports three numbers on the held-out test set:

- **human wrongly flagged** - humans the model called bots (should be low)
- **naive bot caught** - share of naive bots detected (should be high)
- **evasive bot caught** - share of evasive bots detected (this is the one the Breaker attacks)

In [ ]:
def make_model(name):
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0)
    if name == "SVM (RBF)":
        return make_pipeline(StandardScaler(), SVC(class_weight="balanced"))
    if name == "Logistic Regression":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    raise ValueError(name)


def run_once(model_name, features, train_kinds, seed):
    train, test = split(df, seed)
    train = train[train.kind.isin(train_kinds)]
    model = make_model(model_name).fit(train[features], train.is_bot)
    flagged = model.predict(test[features]).astype(bool)          # True = "bot"
    kind = test.kind.to_numpy()
    return {
        "human wrongly flagged": flagged[kind == "human"].mean(),
        "naive bot caught":      flagged[kind == "naive"].mean(),
        "evasive bot caught":    flagged[kind == "evasive"].mean(),
    }


def run(model_name, features, train_kinds):
    runs = pd.DataFrame([run_once(model_name, features, train_kinds, s) for s in range(N_REPEATS)])
    return runs.mean(), runs.std()

## Step 6: the three rounds

Four rows, so we can tell **which change helped**: better features, retraining with evasive bots, or both.
Each cell is the mean over the repeated splits, with the spread after the ± sign.

In [ ]:
BOTH = BASIC_FEATURES + ADVANCED
EXPERIMENTS = [
    ("1  Round 1: basic features, trained on naive bots",    BASIC_FEATURES, ["human", "naive"]),
    ("2  Better features only (trained on naive bots)",      BOTH,           ["human", "naive"]),
    ("3  Retrained with evasive bots, basic features",       BASIC_FEATURES, ["human", "naive", "evasive"]),
    ("4  Round 3: better features + retrained",              BOTH,           ["human", "naive", "evasive"]),
]
METRICS = ["human wrongly flagged", "naive bot caught", "evasive bot caught"]

means, pretty = [], []
for name, feats, kinds in EXPERIMENTS:
    m, s = run("Random Forest", feats, kinds)
    means.append(m)
    pretty.append({"experiment": name, **{k: f"{m[k]:.0%} ± {s[k]:.0%}" for k in METRICS}})

results = pd.DataFrame(means, index=[e[0] for e in EXPERIMENTS])
results.to_csv("results/results_FAKE.csv" if USE_SYNTHETIC else "results/results.csv")
pd.DataFrame(pretty).set_index("experiment")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
width, colors = 0.26, ["#7f7f7f", "#1f77b4", "#ff7f0e"]
x = np.arange(len(EXPERIMENTS))
for j, metric in enumerate(METRICS):
    bars = ax.bar(x + (j - 1) * width, results[metric] * 100, width, label=metric, color=colors[j])
    ax.bar_label(bars, fmt="%.0f%%", fontsize=8, padding=2)
ax.set_xticks(x)
ax.set_xticklabels([f"Row {i + 1}" for i in range(len(EXPERIMENTS))])
ax.set_ylabel("% of test sessions")
ax.set_ylim(0, 108)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8, frameon=False)
title = "Catcher vs Breaker, per round"
if USE_SYNTHETIC:
    title += "  (FAKE DATA - not a result)"
ax.set_title(title)
plt.tight_layout()
plt.savefig("results/rounds_FAKE.png" if USE_SYNTHETIC else "results/rounds.png", dpi=150)
plt.show()

## Step 7: which model? (required comparison for the report)

Same setup, three different classifiers, for the first and the last experiment.
Logistic Regression and SVM need scaled features, so they get a `StandardScaler` in front.

In [ ]:
rows = []
for exp_name, feats, kinds in [EXPERIMENTS[0], EXPERIMENTS[3]]:
    for model_name in ["Random Forest", "SVM (RBF)", "Logistic Regression"]:
        m, _ = run(model_name, feats, kinds)
        rows.append({"experiment": exp_name[:2].strip(), "model": model_name, **{k: f"{m[k]:.0%}" for k in METRICS}})
pd.DataFrame(rows).set_index(["experiment", "model"])

## Step 8: which features matter?

A Random Forest can tell us how much each feature was used. Trained on all sessions here, only to **look** at the
importances (not to measure accuracy).

In [ ]:
all_bots_and_humans = df[df.kind.isin(["human", "naive", "evasive"])]
forest = make_model("Random Forest").fit(all_bots_and_humans[BOTH], all_bots_and_humans.is_bot)
importance = pd.Series(forest.feature_importances_, index=BOTH).sort_values().tail(12)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importance.index, importance.values, color="#1f77b4")
ax.set_xlabel("importance")
ax.set_title("Top features" + ("  (FAKE DATA)" if USE_SYNTHETIC else ""))
plt.tight_layout()
plt.savefig("results/feature_importance_FAKE.png" if USE_SYNTHETIC else "results/feature_importance.png", dpi=150)
plt.show()

## Step 9: what to write in the report (with REAL data)

- **Row 1 vs Row 2/3/4:** if "evasive bot caught" drops a lot from the first row on evasive bots, a detector trained only on naive bots is easy to fool. That is the attack result.
- **Rows 2 and 3 separately:** which one recovers detection, better features or more training data? Row 4 shows both together.
- **"human wrongly flagged"** must stay low in every row. A catcher that flags everybody is useless.
- **Be honest about size:** say how many people and sessions you had, and that the test people were never in training.
- **Logging rhythm trap:** browsers report the mouse every 8-16 ms, but if the Breaker's logs are exactly 16 ms apart, the Catcher can win just by spotting that rhythm, which says nothing about behaviour. Rerun with `EXCLUDE = ["sample_dt_std_ms"]` (Step 1) and report both. Better still, ask the Breaker to log bots at irregular 8-16 ms steps.
- **Limits:** one CAPTCHA style, one input device kind, a small group of people, and an evasion layer written by your teammate (a smarter one could beat Row 4).

Next: set `USE_SYNTHETIC = False`, collect the real data, rerun the notebook, and put `results/rounds.png` on the slides.